In [4]:
import pandas as pd

In [5]:
df = pd.read_parquet('../datasets/modelling_dataset.parquet')

In [6]:
df.head()

,Time_Stamp,Year,Hour,Day_of_Week,Month,Weekend,Holiday,Zone_Int_ID,Amenity,Crossing,...,Wind_Speed(mph),Precipitation(in),Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility Issues,Accident_Count
0,2016-06-14 20:00:00,2016,20,1,6,0,0,0,0.041169,0.233068,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,2016-06-14 20:00:00,2016,20,1,6,0,0,1,0.030181,0.424547,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,2016-06-14 20:00:00,2016,20,1,6,0,0,2,0.000000,0.316667,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,2016-06-14 20:00:00,2016,20,1,6,0,0,3,0.000000,0.161290,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,2016-06-14 20:00:00,2016,20,1,6,0,0,4,0.000000,0.021277,...,9.2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0


# Approach

In this cell, I am employing a 2 Step Approach Using XGBoost

### Step 1: XGB Classifier

Train an XGBoost Classifier on entire dataset to predict a binary target: $0$ (No Accident) vs. $1$ (At least one accident)

### Step 2: XGB Regressor

Filter dataset to only include rows where Accident_Count > 0. Train an XGBoost Regressor on this subset to predict the exact count ($1, 2, 3, \dots, 105$).

### Step 3: Adding Models Together

Model A gives $P(\text{Accident})$. Model B gives $E[\text{Count} \mid \text{Accident}]$. Multiply them together to get  final "Expected Risk Score."

## STEP 1: XBG Classifier

In [7]:
# Creating a binary target variable for accident occurrence to use as target
df["Is_Accident"] = (df["Accident_Count"] > 0).astype(int)

In [8]:
# Looking at where to split the time series data for training, validation, and testing
split_70 = df['Time_Stamp'].quantile(0.70)
split_85 = df['Time_Stamp'].quantile(0.85)

print(f"70% split: {split_70}")
print(f"85% split: {split_85}")

70% split: 2021-03-16 12:00:00
85% split: 2022-03-23 10:00:00


In [9]:
# Creating the datasers for training, validation, and testing based on the time splits
train_df = df[df['Time_Stamp'] < split_70].copy()
val_df = df[(df['Time_Stamp'] >= split_70) & (df['Time_Stamp'] < split_85)].copy()
test_df = df[df['Time_Stamp'] >= split_85].copy()

# Verifying the sizes of the datasets
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Testing set size: {len(test_df)}")

Training set size: 3290824
Validation set size: 705154
Testing set size: 705312


In [10]:
# Subsampling the training set to provide more balance to the classes

accidents = train_df[train_df['Is_Accident'] == 1]
non_accidents = train_df[train_df['Is_Accident'] == 0]

non_accidents_sampled = non_accidents.sample(n=len(accidents) * 5, random_state=42)

train_subsampled = pd.concat([accidents, non_accidents_sampled]).sample(frac=1, random_state=42)

print(f"Subsampled training set size: {len(train_subsampled)}")

Subsampled training set size: 782226


In [11]:
# Having the predictor features for the model excluding year and time stamp
features = ['Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday', 
            'Zone_Int_ID', 'Amenity', 'Crossing', 'Wind_Speed(mph)', 
            'Precipitation(in)', 'Weather_Clear', 'Weather_Cloudy', 
            'Weather_Dust/Windy', 'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 
            'Weather_Stormy', 'Weather_Visibility Issues']

# Creating X and y for the training, validation, and test sets for classification model
X_train_classifier = train_subsampled[features]
y_train_classifier = train_subsampled['Is_Accident']

X_val_classifier = val_df[features]
y_val_classifier = val_df['Is_Accident']

X_test_classifier = test_df[features]
y_test_classifier = test_df['Is_Accident']

In [12]:
from xgboost import XGBClassifier

# Training the model on the subsetted training data
classifier_model = XGBClassifier(
    n_estimators=150, # number of trees to build
    learning_rate=0.05, # how much the model updates prediction
    max_depth=5, #max tree depth
    scale_pos_weight=5,  # model gives 5x more importance to unbalanced class
    objective='binary:logistic', # binary classification
    eval_metric='logloss', # log loss as error
    subsample=0.8, # model randomly samples 80% of train data for each tree
    colsample_bytree=0.8, # model randomly samples 80% of cols fir each tree
    random_state=42 
)

classifier_model.fit(X_train_classifier, y_train_classifier)

,objective,'binary:logistic'
,use_label_encoder,None
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [13]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

# Evaluating the initial model on the validation set 
y_val_preds = classifier_model.predict(X_val_classifier)
y_val_pred_proba = classifier_model.predict_proba(X_val_classifier)[:, 1]

# Checking metrics
print("Classification Report:")
print(classification_report(y_val_classifier, y_val_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_val_classifier, y_val_preds))

print(f"ROC AUC Score: {roc_auc_score(y_val_classifier, y_val_pred_proba):.4f}")

print(f"Average Precision Score: {average_precision_score(y_val_classifier, y_val_pred_proba)}")


Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.73      0.84    665539
           1       0.16      0.88      0.28     39615

    accuracy                           0.74    705154
   macro avg       0.58      0.81      0.56    705154
weighted avg       0.94      0.74      0.81    705154

Confusion Matrix:
[[488405 177134]
 [  4898  34717]]
ROC AUC Score: 0.8833
Average Precision Score: 0.30163457104849106


In [14]:
import numpy as np
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import fbeta_score, make_scorer

X_tuning_classifier = pd.concat([X_train_classifier, X_val_classifier])
y_tuning_classifier = pd.concat([y_train_classifier, y_val_classifier])

# Identifying which is used for training (-1) and which is used for validation(0)
split_index = np.full(X_tuning_classifier.shape[0], -1)
split_index[len(train_subsampled):] = 0
pds = PredefinedSplit(test_fold=split_index)

# GridSearchCV parameters
param_grid = {
    'max_depth': [5, 7],
    'learning_rate': [0.1, 0.05],
    'n_estimators': [150, 200, 225],
    'scale_pos_weight': [5, 10, 15],
    'max_delta_step': [1, 2],
}

# Use f2 to prioritize recall while still considering precision
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=1)

grid_search = GridSearchCV(
    estimator=XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    cv=pds,
    scoring="average_precision",  
    verbose=1
)

grid_search.fit(X_tuning_classifier, y_tuning_classifier)

best_classifier = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

Fitting 1 folds for each of 72 candidates, totalling 72 fits


/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 400, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 90, in _cached_call


KeyboardInterrupt: 

In [ ]:
# Determining Optimal Threshold for Classification
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Using the best classifier on the validation set
y_val_probs = best_classifier.predict_proba(val_df[features])[:, 1]
precision, recall, thresholds = precision_recall_curve(val_df['Is_Accident'], y_val_probs)

# Plotting precision recall curve for visual analysis
plt.figure(figsize=(8, 4))
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")
plt.xlabel("Threshold")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Testing final model on test dataset
y_test_pred_proba = best_classifier.predict_proba(X_test_classifier)[:, 1]

# Use a 0.3 threshold based on above precision-recall curve analysis 
y_test_preds = (y_test_pred_proba >= 0.4).astype(int)

# Checking metrics
print("Classification Report:")
print(classification_report(y_test_classifier, y_test_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test_classifier, y_test_preds))

print(f"ROC AUC Score: {roc_auc_score(y_test_classifier, y_test_pred_proba):.4f}")

print(f"Average Precision Score: {average_precision_score(y_test_classifier, y_test_pred_proba)}")


## STEP 2: XGB Regressor

In [ ]:
# Looking at only accidents to train regression model
train_accidents = train_df[train_df['Is_Accident'] == 1].copy()
val_accidents = val_df[val_df['Is_Accident'] == 1].copy()

X_train_regressor = train_accidents[features]
y_train_regressor = train_accidents['Accident_Count'] # Use accident count as target for regression model

X_val_regressor = val_accidents[features]
y_val_regressor = val_accidents['Accident_Count']

# Using log of values to remove large outliers
y_train_log = np.log1p(y_train_regressor) 
y_val_log = np.log1p(y_val_regressor)

In [ ]:
# Doing hyperparameter tuning for regression model using the same predefined split as classification model 

from xgboost import XGBRegressor

X_tuning_regressor = pd.concat([X_train_regressor, X_val_regressor])
y_tuning_regressor = pd.concat([y_train_regressor, y_val_regressor])
y_tuning_log = pd.concat([y_train_log, y_val_log])

split_index_b = np.full(X_tuning_regressor.shape[0], -1)
split_index_b[len(train_accidents):] = 0
pds_b = PredefinedSplit(test_fold=split_index_b)

param_grid_regressor = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 150, 200],
}

grid_search_regressor = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
    param_grid=param_grid_regressor,
    cv=pds_b,
    scoring='neg_mean_absolute_error', 
    verbose=1
)

grid_search_regressor.fit(X_tuning_regressor, y_tuning_log)
best_regressor = grid_search_regressor.best_estimator_
print(f"Best Regressor Parameters: {grid_search_regressor.best_params_}")

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Evaluating the best regression model on the validation set of accidents
y_val_log_preds = best_regressor.predict(val_accidents[features])
y_val_final_preds = np.expm1(y_val_log_preds)

# Calculating evaluation metrics
mae = mean_absolute_error(val_accidents['Accident_Count'], y_val_final_preds)
rmse = root_mean_squared_error(val_accidents['Accident_Count'], y_val_final_preds)
r2 = r2_score(val_accidents['Accident_Count'], y_val_final_preds)

print(f"Mean Absolute Error: {mae}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared: {r2}")

## STEP 3: Final Synthesis

In [ ]:
# Using the classifier to predict the probability of accident on test
test_probs = best_classifier.predict_proba(X_test_classifier)[:, 1]
test_preds = (test_probs >= 0.4).astype(int)

# Using the regressor to predict the numbver of accidents on test
test_log_severity = best_regressor.predict(X_test_classifier)
test_severity = np.expm1(test_log_severity)


final_results = test_df.copy() 
final_results['Prob_Accident'] = test_probs
final_results['Predicted_Accident'] = test_preds
final_results['Expected_Severity'] = test_severity

# Creating the final risk score by multiplying the probability of an accident occurring with the expected number of accidents
final_results['Risk_Score'] = test_probs * test_severity
final_results["Risk_Score_2"] = test_preds * test_severity

In [ ]:
final_results[final_results['Accident_Count'] > 1]

## STEP 4: Visualizing Results

#### Looking at the actual accidents in the test set

In [ ]:
import plotly.express as px
import h3

# Aggregate actual accident counts from test set by zone
miami_test = final_results[final_results['City_Miami'] == 1].copy()

zone_accidents = miami_test.groupby('Zone_ID')['Accident_Count'].sum().reset_index()
zone_accidents.rename(columns={'Accident_Count': 'Total_Accidents'}, inplace=True)

# Build GeoJSON from H3 zone IDs
features = []
for zone in zone_accidents['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plot the map
fig = px.choropleth_map(
    zone_accidents,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Total_Accidents',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    #center={'lat': 29.7604, 'lon': -95.3698},
    center={'lat': 25.7617, 'lon': -80.1918}, #for Miami
    title='Test Set: Actual Accident Counts by Zone',
    labels={'Total_Accidents': 'Accidents'}
)

fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

In [ ]:
import plotly.express as px
import h3

# Aggregate predicted risk scores from test set by zone
miami_test = final_results[final_results['City_Miami'] == 1].copy()

zone_risk = miami_test.groupby('Zone_ID')['Risk_Score'].sum().reset_index()
zone_risk.rename(columns={'Risk_Score': 'Total_Risk_Score'}, inplace=True)

# Build GeoJSON from H3 zone IDs
features = []
for zone in zone_risk['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plot the map
fig = px.choropleth_map(
    zone_risk,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Total_Risk_Score',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    #center={'lat': 29.7604, 'lon': -95.3698},
    center={'lat': 25.7617, 'lon': -80.1918}, #for Miami
    title='Test Set: Predicted Risk Score by Zone',
    labels={'Total_Risk_Score': 'Risk Score'}
)

fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()